<a href="https://colab.research.google.com/github/sarshadad-codeee/FlyRank_ML_Task1/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [15]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/sarshadad-codeee/FlyRank_ML_Task1"
REPO_DIR = "FlyRank_ML_Task1"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working directory:", os.getcwd())

Working directory: /content/FlyRank_ML_Task1/FlyRank_ML_Task1/FlyRank_ML_Task1


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [16]:
"""
Lane: Lane 2 — Refresh / Content Opportunity Scoring

Task type: Ranking / scoring.

Per the framing-ml-problems skill's task-type table: my question is
"which pages should a reviewer look at first?" — that maps directly to
"Which ones first?" -> Ranking/scoring, not plain classification.

In practice this is built in two steps, matching what I already did in
Assignment 1 Notebook 2: a classifier estimates a probability (e.g. "will
this page keep declining?"), and that probability becomes the ranking
score used to sort the full page inventory. So classification is the
mechanism, but ranking/scoring is the actual task type and what the
output is used for — a reviewer doesn't care about the raw probability,
they care about the ORDER of the queue.
"""

'\nLane: Lane 2 — Refresh / Content Opportunity Scoring\n\nTask type: Ranking / scoring.\n\nPer the framing-ml-problems skill\'s task-type table: my question is\n"which pages should a reviewer look at first?" — that maps directly to\n"Which ones first?" -> Ranking/scoring, not plain classification.\n\nIn practice this is built in two steps, matching what I already did in\nAssignment 1 Notebook 2: a classifier estimates a probability (e.g. "will\nthis page keep declining?"), and that probability becomes the ranking\nscore used to sort the full page inventory. So classification is the\nmechanism, but ranking/scoring is the actual task type and what the\noutput is used for — a reviewer doesn\'t care about the raw probability,\nthey care about the ORDER of the queue.\n'

In [17]:
"""
Action this output supports: the ranked score doesn't stop at "this page
is a 1 or a 0" — it feeds a queue where a content reviewer takes one of
several concrete actions per flagged page: refresh/update stale content,
fix low CTR via title/meta changes, investigate a real decline, or
monitor with no action yet. The reason codes (e.g. stale_visible_page,
declining_with_demand) tell the reviewer WHICH action likely applies,
not just THAT the page is worth a look.
"""

'\nAction this output supports: the ranked score doesn\'t stop at "this page\nis a 1 or a 0" — it feeds a queue where a content reviewer takes one of\nseveral concrete actions per flagged page: refresh/update stale content,\nfix low CTR via title/meta changes, investigate a real decline, or\nmonitor with no action yet. The reason codes (e.g. stale_visible_page,\ndeclining_with_demand) tell the reviewer WHICH action likely applies,\nnot just THAT the page is worth a look.\n'

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [18]:
"""
Target/proxy: is_declining_label, derived from trend_direction == "down"
(same as the starter pipeline's label).

Honesty check (per framing-ml-problems' "two rules" and the flyrank-data
label trap): this target IS an observed measurement — trend_direction is
computed from trend_pct, which reflects real trailing search-volume
movement, not a FlyRank product decision rule like health_score. So it's
allowed as a TARGET. What it is NOT allowed to be is a FEATURE — if I ever
feed trend_direction or trend_pct into the model as an input, the model
would just be predicting itself.

Known weakness: this is a CURRENT-window bucket (declining right now),
not a FUTURE-observed outcome. The lane guide calls this a "beginner
proxy label." A stronger version — which I'll build once I have a proper
data contract with clean feature/target windows — would be:

    features from prior 90 days -> decline over the NEXT 30 days

For this framing pass, I'm using the current-window proxy because it's
what's available and verifiable in the starter dataset today, but I'm
flagging it explicitly as provisional, not final.
"""

'\nTarget/proxy: is_declining_label, derived from trend_direction == "down"\n(same as the starter pipeline\'s label).\n\nHonesty check (per framing-ml-problems\' "two rules" and the flyrank-data\nlabel trap): this target IS an observed measurement — trend_direction is\ncomputed from trend_pct, which reflects real trailing search-volume\nmovement, not a FlyRank product decision rule like health_score. So it\'s\nallowed as a TARGET. What it is NOT allowed to be is a FEATURE — if I ever\nfeed trend_direction or trend_pct into the model as an input, the model\nwould just be predicting itself.\n\nKnown weakness: this is a CURRENT-window bucket (declining right now),\nnot a FUTURE-observed outcome. The lane guide calls this a "beginner\nproxy label." A stronger version — which I\'ll build once I have a proper\ndata contract with clean feature/target windows — would be:\n\n    features from prior 90 days -> decline over the NEXT 30 days\n\nFor this framing pass, I\'m using the current-window 

## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [19]:
"""
Success metric: Precision@K, specifically Precision@50.

Why this metric and not accuracy or plain ROC-AUC: a reviewer only has
capacity to act on a fixed number of pages (say, the top 50 in the
queue), so what matters is "of the top 50 pages we rank first, how many
actually deserved review?" — not how the model performs across all
30,000 pages, most of which nobody will ever look at.

I'll also track recall on high-value pages as a secondary check, since
Assignment 1's framing already established that false negatives (a
declining page that never gets flagged) are costlier than false
positives (a reviewer wastes time on a page that didn't need it) — a
metric that only rewards precision could hide that risk.

Baseline to beat: from Assignment 1's starter pipeline results, the
hand-written rule scored Precision@50 = 0.240, and the random forest
scored 0.740. Any model I build for this lane needs to be benchmarked
against that same baseline logic, not judged in isolation.
"""

'\nSuccess metric: Precision@K, specifically Precision@50.\n\nWhy this metric and not accuracy or plain ROC-AUC: a reviewer only has\ncapacity to act on a fixed number of pages (say, the top 50 in the\nqueue), so what matters is "of the top 50 pages we rank first, how many\nactually deserved review?" — not how the model performs across all\n30,000 pages, most of which nobody will ever look at.\n\nI\'ll also track recall on high-value pages as a secondary check, since\nAssignment 1\'s framing already established that false negatives (a\ndeclining page that never gets flagged) are costlier than false\npositives (a reviewer wastes time on a page that didn\'t need it) — a\nmetric that only rewards precision could hide that risk.\n\nBaseline to beat: from Assignment 1\'s starter pipeline results, the\nhand-written rule scored Precision@50 = 0.240, and the random forest\nscored 0.740. Any model I build for this lane needs to be benchmarked\nagainst that same baseline logic, not judged in iso

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [20]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Unit of analysis: one row = one content page (content_id),
# summarized over its trailing 90-day window.
print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"One row = one page. Unique content_id count: {df['content_id'].nunique()}")

# Show the lane-relevant slice of columns — the signals this lane cares about
lane_cols = [
    "content_id", "client_id", "impressions_90d", "days_since_last_update",
    "avg_position", "content_age_days", "word_count", "ctr",
    "engagement_rate", "trend_direction"
]
df[lane_cols].head(10)

Shape: 30000 rows, 44 columns
One row = one page. Unique content_id count: 30000


,content_id,client_id,impressions_90d,days_since_last_update,avg_position,content_age_days,word_count,ctr,engagement_rate,trend_direction
0,content_304f48230142,client_f369cb89fc,3803,20,10.6,187,3221.0,0.76,5.88,down
1,content_a1fb4e703a9e,client_4e07408562,15320,25,20.3,445,2481.0,0.05,0.00,down
2,content_9aa793d4d895,client_7f2253d7e2,12581,20,36.5,141,3515.0,0.09,0.00,down
3,content_331d6c4de07b,client_19581e27de,11751,22,6.2,463,NaN,0.49,1.28,stable
4,content_d99b7a2d90ca,client_3fdba35f04,19140,14,44.0,263,2803.0,0.13,0.00,down
5,content_d4084a4bc775,client_f369cb89fc,3970,20,8.5,147,3080.0,0.03,0.00,down
6,content_9a34b442b552,client_8722616204,20,20,7.0,90,3059.0,0.00,0.00,down
7,content_a63219c6e95a,client_19581e27de,1724,22,21.2,445,NaN,0.06,3.57,stable
8,content_5e6c160719bc,client_6208ef0f77,32574,20,46.0,90,3807.0,0.09,5.88,down
9,content_c27558df2b0c,client_19581e27de,1240,104,4.9,257,NaN,0.16,0.00,down


In [21]:
# Sketch what the target column would look like
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print(f"Target distribution:\n{df['is_declining_label'].value_counts()}")
print(f"\nBase rate (declining): {df['is_declining_label'].mean():.1%}")
print("\nSample rows with target attached:")
df[lane_cols + ["is_declining_label"]].head(10)

Target distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Base rate (declining): 54.2%

Sample rows with target attached:


,content_id,client_id,impressions_90d,days_since_last_update,avg_position,content_age_days,word_count,ctr,engagement_rate,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,3803,20,10.6,187,3221.0,0.76,5.88,down,1
1,content_a1fb4e703a9e,client_4e07408562,15320,25,20.3,445,2481.0,0.05,0.00,down,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,20,36.5,141,3515.0,0.09,0.00,down,1
3,content_331d6c4de07b,client_19581e27de,11751,22,6.2,463,NaN,0.49,1.28,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,14,44.0,263,2803.0,0.13,0.00,down,1
5,content_d4084a4bc775,client_f369cb89fc,3970,20,8.5,147,3080.0,0.03,0.00,down,1
6,content_9a34b442b552,client_8722616204,20,20,7.0,90,3059.0,0.00,0.00,down,1
7,content_a63219c6e95a,client_19581e27de,1724,22,21.2,445,NaN,0.06,3.57,stable,0
8,content_5e6c160719bc,client_6208ef0f77,32574,20,46.0,90,3807.0,0.09,5.88,down,1
9,content_c27558df2b0c,client_19581e27de,1240,104,4.9,257,NaN,0.16,0.00,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [22]:
"""
A fixed if-statement rule could catch ONE obvious pattern at a time —
e.g. "flag pages with impressions > 500 AND days_since_last_update >
180." That's exactly what the starter baseline_refresh_score already
does, using four separate weighted rules combined by hand-picked
weights (0.40 / 0.30 / 0.25 / 0.05).

The problem: those weights were chosen by guesswork, not evidence, and
they can't adapt when signals interact in non-obvious ways — e.g. maybe
freshness only matters when position is already weak, or maybe CTR gaps
matter more for one content_type than another. A hand-written rule
can't discover or represent that kind of conditional interaction across
6+ correlated signals at once.

This isn't hypothetical — Assignment 1 already measured the gap on this
exact data: the hand rule scored Precision@50 = 0.240, the random forest
scored 0.740. That's not a marginal improvement, it's finding real
structure the rule missed. ML earns its place here specifically because
the framing-ml-problems skill's condition is met: "the pattern is real
but too messy to write by hand — many signals, tangled, shifting over
time." A dashboard or a single rule genuinely wouldn't capture this.
"""

'\nA fixed if-statement rule could catch ONE obvious pattern at a time —\ne.g. "flag pages with impressions > 500 AND days_since_last_update >\n180." That\'s exactly what the starter baseline_refresh_score already\ndoes, using four separate weighted rules combined by hand-picked\nweights (0.40 / 0.30 / 0.25 / 0.05).\n\nThe problem: those weights were chosen by guesswork, not evidence, and\nthey can\'t adapt when signals interact in non-obvious ways — e.g. maybe\nfreshness only matters when position is already weak, or maybe CTR gaps\nmatter more for one content_type than another. A hand-written rule\ncan\'t discover or represent that kind of conditional interaction across\n6+ correlated signals at once.\n\nThis isn\'t hypothetical — Assignment 1 already measured the gap on this\nexact data: the hand rule scored Precision@50 = 0.240, the random forest\nscored 0.740. That\'s not a marginal improvement, it\'s finding real\nstructure the rule missed. ML earns its place here specifically be

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.